# PINN-1 — Water Balance / Mass Conservation PINN
**Caspian Sea — DAHITI water level   ·   ERA5 / GLDAS forcings   ·   Volga discharge**

The simplest and most fundamental of the three PINNs.

| | **PINN-1 (this notebook)** | PINN-2 | PINN-3 |
|---|---|---|---|
| Closure law | **Mass conservation** | Budyko (long-term ET) | Penman energy balance |
| Soft constraint | **dh = (P − α·E) + β·Q/A + γ·(P_b − PET_b)** | ET = φ/(1+φⁿ)^(1/n) · P | E = α·Δ/(Δ+γ)·Rn + … |
| Learnable physics | α_E, β_volga, γ_other (3) | n_t, scale_t (2) | α_PT, C_wind, γ_runoff (3) |
| Architecture | CNN-LSTM | CNN-LSTM | CNN-LSTM |

Same architecture, same hyperparameters, same data, same train/test split as
PINN-2 and PINN-3 — only the physics loss differs.

**Why this PINN exists.** PINN-1 doesn't try to *predict* evaporation (PINN-3's
job) or *partition* runoff (PINN-2's job). It just enforces the universally
true statement: *what comes in minus what goes out equals the level change*.
Of the three PINNs, this is the constraint with the least room to be wrong —
which is exactly why we expect it to be the most robust.

---


## 1.  Imports and configuration

In [1]:
import os
import json
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import matplotlib.pyplot as plt

# pinn1_water_balance.py must sit next to this notebook
from pinn1_water_balance import WaterBalancePINN1


In [2]:
# ── Configuration ────────────────────────────────────────────────────────
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_END_YEAR = 2017          # same split as PINN-2 and PINN-3
BATCH_SIZE     = 16
LR             = 3e-4
EPOCHS         = 300
SEQ_LEN        = 6
LAMBDA_PHYS    = 1.0
PATIENCE       = 50

VERSION = "v20_deep_learning_monthly"
OUT_DIR = f"results/pinn1/{VERSION}"
os.makedirs(OUT_DIR, exist_ok=True)

# Paths — files alongside the notebook (no data/ subfolder)
DATA_PATH  = "merged_grid_dataset_monthly.nc"
VOLGA_PATH = "volga_discharge.csv"

print(f"Device: {DEVICE} | Version: {VERSION} | SeqLen: {SEQ_LEN}")


Device: cpu | Version: v20_deep_learning_monthly | SeqLen: 6


## 2.  Data loading

`lsm` (land-sea mask) is inside `merged_grid_dataset_monthly.nc` — no
separate file needed.


In [3]:
ds = xr.open_dataset(DATA_PATH)
print("Variables :", list(ds.data_vars))
print("Time range:", str(ds.time.values[0])[:10], "→", str(ds.time.values[-1])[:10])

volga_df         = pd.read_csv(VOLGA_PATH)
volga_df['time'] = pd.to_datetime(volga_df['time'])
volga_df         = volga_df.set_index('time')

# Land-sea mask from the same NetCDF
lsm_raw = ds.lsm.values
if lsm_raw.ndim == 3:
    lsm_raw = lsm_raw[0]
basin_mask = lsm_raw >  0.5
lake_mask  = lsm_raw <= 0.5
print(f"Grid: {basin_mask.shape}   basin={basin_mask.sum()}   lake={lake_mask.sum()}")


Variables : ['u10', 'v10', 'd2m', 't2m', 'snowc', 'sde', 'sd', 'sp', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 'tp', 'e', 'ssr', 'str', 'Qs_acc', 'Qsb_acc', 'SWE_inst', 'water_level', 'delta_water_level', 'lsm']
Time range: 1993-01-01 → 2025-12-01
Grid: (110, 90)   basin=5506   lake=4394


## 3.  Basin- and lake-averaged forcings

PINN-1's water balance needs five scalar fluxes per month:
* **P_lake**, **E_lake** — lake surface fluxes (from ERA5 directly, no Penman)
* **P_basin**, **PET_basin** — for the residual basin term
* (Volga Q is loaded separately from CSV, exposed by the Dataset class)


In [4]:
def basin_avg(field, mask):
    return field.where(mask).mean(dim=['latitude', 'longitude']).to_series().fillna(0)

# Basin scalars (Ural / Terek / residual catchment)
P_basin   = basin_avg(ds.tp, basin_mask)
PET_basin = (basin_avg(ds.ssr, basin_mask) / 2.5e9).clip(lower=1e-3)

# Lake scalars: directly observed P and E from ERA5
P_lake = basin_avg(ds.tp,  lake_mask)
E_lake = basin_avg(-ds.e,  lake_mask)         # ERA5 e is negative → flip sign

# Target
water_level = ds.water_level.to_series().ffill().fillna(0)
delta_h     = water_level.diff().fillna(0)

print(f"P_basin   mean = {P_basin.mean():.4f} m/month")
print(f"PET_basin mean = {PET_basin.mean():.4f} m/month")
print(f"P_lake    mean = {P_lake.mean():.4f} m/month")
print(f"E_lake    mean = {E_lake.mean():.4f} m/month")
print(f"Δh        mean = {delta_h.mean():.5f} m/month   (negative = lake shrinking)")


P_basin   mean = 0.0377 m/month
PET_basin mean = 0.1489 m/month
P_lake    mean = 0.0255 m/month
E_lake    mean = 0.0853 m/month
Δh        mean = -0.00607 m/month   (negative = lake shrinking)


## 4.  Dataset class

13-channel gridded input + 5 scalar physics drivers (P_lake, E_lake, Q_volga,
P_basin, PET_basin). Volga discharge appears both as input channel 13 (broadcast
across the grid) AND as a scalar for the water-balance loss.


In [5]:
class WaterBalanceDataset(Dataset):
    """Gridded ERA5/GLDAS fields + scalar inflows/outflows for PINN-1."""

    GRID_VARS = ['tp', 't2m', 'ssr', 'str', 'sp', 'e', 'd2m',
                 'u10', 'v10', 'Qs_acc', 'Qsb_acc', 'SWE_inst']

    def __init__(self, ds, volga_df, seq_len=6, stats=None):
        self.times   = ds.time.values
        self.seq_len = seq_len

        # Gridded tensor + Volga as grid channel AND scalar series
        data_list = [ds[v].values for v in self.GRID_VARS]
        volga_grid, volga_series = [], []
        for t in ds.time.values:
            t_dt = pd.to_datetime(t)
            try:
                q = volga_df.loc[t_dt:t_dt]['volga_q'].values[0]
            except Exception:
                q = 8000.0
            volga_grid.append(np.full_like(data_list[0][0], q, dtype=np.float32))
            volga_series.append(float(q))
        data_list.append(np.array(volga_grid))
        self.data    = np.stack(data_list, axis=1)
        self.Q_volga = np.array(volga_series, dtype=np.float32)

        # Scalars for the physics loss
        self.targets   = delta_h.values
        self.p_lake    = P_lake.values
        self.e_lake    = E_lake.values
        self.p_basin   = P_basin.values
        self.pet_basin = PET_basin.values

        if stats is None:
            self.stats = {
                'mean': np.nanmean(self.data, axis=(0, 2, 3), keepdims=True),
                'std':  np.nanstd(self.data,  axis=(0, 2, 3), keepdims=True),
            }
            self.target_mean = float(np.mean(self.targets))
            self.target_std  = float(np.std(self.targets))
        else:
            self.stats       = stats
            self.target_mean = stats['target_mean']
            self.target_std  = stats['target_std']

        self.data         = np.nan_to_num((self.data - self.stats['mean']) /
                                          (self.stats['std'] + 1e-8))
        self.targets_raw  = self.targets
        self.targets_norm = (self.targets - self.target_mean) / (self.target_std + 1e-8)

        self.valid_indices = [i for i in range(len(self.times)) if i >= seq_len - 1]

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        i = self.valid_indices[idx]
        x = self.data[i - self.seq_len + 1 : i + 1]
        return (
            torch.FloatTensor(x),
            torch.FloatTensor([self.targets_norm[i]]),
            torch.FloatTensor([self.targets_raw[i]]),
            torch.FloatTensor([self.p_lake[i]]),
            torch.FloatTensor([self.e_lake[i]]),
            torch.FloatTensor([self.Q_volga[i]]),
            torch.FloatTensor([self.p_basin[i]]),
            torch.FloatTensor([self.pet_basin[i]]),
        )


## 5.  Chronological train / test split

In [6]:
all_years = pd.to_datetime(ds.time.values).year
full_ds   = WaterBalanceDataset(ds, volga_df, seq_len=SEQ_LEN)

train_indices = [i for i, v in enumerate(full_ds.valid_indices) if all_years[v] <= TRAIN_END_YEAR]
test_indices  = [i for i, v in enumerate(full_ds.valid_indices) if all_years[v] >  TRAIN_END_YEAR]

train_subset = Subset(full_ds, train_indices)
test_subset  = Subset(full_ds, test_indices)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_subset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train months: {len(train_indices)}  ({all_years.min()}–{TRAIN_END_YEAR})")
print(f"Test  months: {len(test_indices)}   ({TRAIN_END_YEAR+1}–{all_years.max()})")


Train months: 295  (1993–2017)
Test  months: 96   (2018–2025)


## 6.  Model, optimiser, scheduler

In [7]:
model     = WaterBalancePINN1(in_channels=13, seq_len=SEQ_LEN).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

n_params = sum(p.numel() for p in model.parameters())
print(f"PINN-1 parameters: {n_params:,}")
print(f"Initial physics params (bias init): α_E≈1.0  β_volga≈1.0  γ_other≈0.3")


PINN-1 parameters: 339,012
Initial physics params (bias init): α_E≈1.0  β_volga≈1.0  γ_other≈0.3


## 7.  Training loop

Loss = **data loss (MSE on normalised Δh)** + λ · **water-balance physics loss (MSE on raw Δh, m²)**.
λ = 1.0, matching PINN-2 and PINN-3 conventions.


In [8]:
print(f"\n== Training PINN-1 ({VERSION}) ==")
history = {
    'data_loss': [], 'phys_loss': [], 'val_loss': [],
    'alpha_E': [], 'beta_volga': [], 'gamma_other': [],
    'Q_in_m': [], 'lake_contrib': [],
}

best_val_loss    = float('inf')
patience_counter = 0
best_model_state = None

t_mean = full_ds.target_mean
t_std  = full_ds.target_std

for epoch in range(EPOCHS):
    model.train()
    ep_data_raw = ep_phys_raw = 0.0
    ep_a = ep_b = ep_g = 0.0
    ep_Q = ep_lc = 0.0

    for batch in train_loader:
        (x, y_norm, y_raw,
         p_l, e_l, Q_v, p_b, pet_b) = [b.to(DEVICE) for b in batch]

        optimizer.zero_grad()

        dh_pred_norm, alpha_E, beta_volga, gamma_other = model(x)
        dh_pred_raw  = dh_pred_norm * t_std + t_mean

        loss_data = F.mse_loss(dh_pred_norm, y_norm)
        loss_phys, diag = model.water_balance_loss(
            dh_pred_raw, alpha_E, beta_volga, gamma_other,
            p_l, e_l, Q_v, p_b, pet_b,
        )

        total_loss = loss_data + LAMBDA_PHYS * loss_phys
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        with torch.no_grad():
            data_raw = F.mse_loss(dh_pred_raw, y_raw).item()
        ep_data_raw += data_raw
        ep_phys_raw += loss_phys.item()
        ep_a  += alpha_E.mean().item()
        ep_b  += beta_volga.mean().item()
        ep_g  += gamma_other.mean().item()
        ep_Q  += diag['Q_in_m']
        ep_lc += diag['lake_contrib']

    scheduler.step()

    # Validation
    model.eval()
    val_l_raw = 0.0
    with torch.no_grad():
        for batch in test_loader:
            x, _, y_raw, *_ = [b.to(DEVICE) for b in batch]
            dh_pred_norm, *_ = model(x)
            dh_pred_raw = dh_pred_norm * t_std + t_mean
            val_l_raw  += F.mse_loss(dh_pred_raw, y_raw).item()

    n_b = len(train_loader)
    avg_data = ep_data_raw / n_b
    avg_phys = ep_phys_raw / n_b
    avg_val  = val_l_raw / len(test_loader)
    avg_a, avg_b, avg_g = ep_a/n_b, ep_b/n_b, ep_g/n_b
    avg_Q, avg_lc       = ep_Q/n_b, ep_lc/n_b

    for k, v in zip(
        ['data_loss','phys_loss','val_loss','alpha_E','beta_volga','gamma_other','Q_in_m','lake_contrib'],
        [avg_data, avg_phys, avg_val, avg_a, avg_b, avg_g, avg_Q, avg_lc],
    ):
        history[k].append(v)

    if avg_val < best_val_loss:
        best_val_loss    = avg_val
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0:
        print(f"Ep {epoch+1:3d}/{EPOCHS} | "
              f"Data(m²): {avg_data:.6f} | Phys(m²): {avg_phys:.6f} | Val(m²): {avg_val:.6f} | "
              f"α_E={avg_a:.2f}  β_v={avg_b:.2f}  γ_o={avg_g:.2f}  Q={avg_Q:.3f}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)



== Training PINN-1 (v20_deep_learning_monthly) ==
Ep  10/300 | Data(m²): 0.003155 | Phys(m²): 0.001495 | Val(m²): 0.002480 | α_E=0.96  β_v=1.03  γ_o=0.26  Q=0.058
Ep  20/300 | Data(m²): 0.002679 | Phys(m²): 0.001119 | Val(m²): 0.002467 | α_E=0.93  β_v=1.05  γ_o=0.25  Q=0.060
Ep  30/300 | Data(m²): 0.002680 | Phys(m²): 0.001149 | Val(m²): 0.002522 | α_E=0.91  β_v=0.99  γ_o=0.23  Q=0.056
Ep  40/300 | Data(m²): 0.002590 | Phys(m²): 0.001081 | Val(m²): 0.002598 | α_E=0.85  β_v=0.96  γ_o=0.23  Q=0.055
Ep  50/300 | Data(m²): 0.002472 | Phys(m²): 0.001040 | Val(m²): 0.002565 | α_E=0.85  β_v=0.92  γ_o=0.22  Q=0.052
Ep  60/300 | Data(m²): 0.002479 | Phys(m²): 0.001018 | Val(m²): 0.002473 | α_E=0.83  β_v=0.90  γ_o=0.21  Q=0.051
Early stopping at epoch 67


## 8.  Loss curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history['data_loss'], label='Data Loss (Raw)',  color='#00a8ff')
plt.plot(history['phys_loss'], label='Water-Balance Phys Loss', color='#ff9f43')
plt.plot(history['val_loss'],  label='Val Loss',         color='#2ecc71')
plt.yscale('log'); plt.title(f"PINN-1 {VERSION} — Loss History")
plt.legend(); plt.grid(True, alpha=0.2)
plt.savefig(os.path.join(OUT_DIR, 'loss_curves.png'), dpi=130, bbox_inches='tight')
plt.show()


## 9.  Test-set water-level reconstruction

In [ ]:
model.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        x, _, y_raw, *_ = [b.to(DEVICE) for b in batch]
        p_norm, *_ = model(x)
        p_raw = p_norm * t_std + t_mean
        test_preds  .append(p_raw.cpu().numpy().flatten())
        test_targets.append(y_raw.cpu().numpy().flatten())

p_raw = np.concatenate(test_preds)
t_raw = np.concatenate(test_targets)
test_times = pd.to_datetime([full_ds.times[full_ds.valid_indices[i]] for i in test_indices])

last_train_idx    = train_indices[-1]
start_test_level  = water_level.iloc[full_ds.valid_indices[last_train_idx]]
actual_test_level = start_test_level + np.cumsum(t_raw)
pred_test_level   = start_test_level + np.cumsum(p_raw)

plt.figure(figsize=(12, 5))
plt.plot(test_times, actual_test_level, label='Actual (Test Region)', color='#00a8ff', linewidth=2)
plt.plot(test_times, pred_test_level,   label='PINN-1 Prediction',    color='#ff4d4d', linestyle='--', linewidth=2)
plt.title(f"PINN-1 {VERSION}: Test Set Performance", fontsize=14)
plt.ylabel("Water Level (m)"); plt.grid(True, alpha=0.3); plt.legend()
plt.savefig(os.path.join(OUT_DIR, 'water_level_test.png'), dpi=130, bbox_inches='tight')
plt.show()


## 10.  Full-period reconstruction

In [ ]:
full_times  = pd.to_datetime([full_ds.times[v] for v in full_ds.valid_indices])
full_actual = water_level.iloc[full_ds.valid_indices].values

plt.figure(figsize=(14, 6))
plt.plot(full_times, full_actual, label='Actual Water Level', color='#00a8ff', linewidth=1.5, alpha=0.7)
plt.plot(test_times, pred_test_level, label='PINN-1 Test Prediction',
         color='#ff4d4d', linestyle='--', linewidth=2.5)
plt.axvline(test_times[0], color='black', linestyle='-', alpha=0.5)
plt.fill_between(full_times, full_actual.min()-0.5, full_actual.max()+0.5,
                 where=(full_times < test_times[0]),
                 color='gray', alpha=0.1, label='Training Period')
plt.fill_between(test_times, full_actual.min()-0.5, full_actual.max()+0.5,
                 color='blue', alpha=0.05, label='Test Period')
plt.title(f"PINN-1 {VERSION}: Full Period Reconstruction", fontsize=14)
plt.ylabel("Water Level (m)"); plt.grid(True, alpha=0.3); plt.legend(loc='lower left')
plt.savefig(os.path.join(OUT_DIR, 'full_period_reconstruction.png'), dpi=130, bbox_inches='tight')
plt.show()


## 11.  Learned physics-correction trajectories

Three corrections to watch — interpretable defence material:

* **α_E** should sit near 1.0. If it drifts low (<0.7) the network thinks ERA5
  overestimates evaporation in this basin; if it drifts high (>1.3) ERA5 is
  underestimating.

* **β_volga** should also sit near 1.0. Significantly below 1.0 implies that
  groundwater interception / channel losses reduce the effective Volga
  contribution — a known phenomenon in the Volga delta.

* **γ_other** soaks up everything else (Ural, Terek, smaller tributaries,
  groundwater). Expect 0.2–0.6.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, key, color, label in zip(
    axes,
    ['alpha_E', 'beta_volga', 'gamma_other'],
    ['#ff4d4d', '#9b59b6', '#2ecc71'],
    ['α_E (ERA5 evap. scaling)', 'β_volga (Volga transfer)', 'γ_other (residual basin)'],
):
    ax.plot(history[key], color=color, linewidth=2)
    ax.set_title(f"Mean {label} per epoch")
    ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'physics_params.png'), dpi=130, bbox_inches='tight')
plt.show()


## 12.  Metrics + JSON

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae  = mean_absolute_error(t_raw, p_raw)
rmse = float(np.sqrt(np.mean((p_raw - t_raw) ** 2)))
r2   = r2_score(t_raw, p_raw)
corr = float(np.corrcoef(p_raw, t_raw)[0, 1])

mae_level  = mean_absolute_error(actual_test_level, pred_test_level)
rmse_level = float(np.sqrt(np.mean((actual_test_level - pred_test_level) ** 2)))
r2_level   = r2_score(actual_test_level, pred_test_level)

metrics = {
    'pinn'              : 'PINN-1 (Water Balance / Mass Conservation)',
    'version'           : VERSION,
    'mae_dh_test'       : float(mae),
    'rmse_dh_test'      : rmse,
    'r2_dh_test'        : float(r2),
    'corr_dh_test'      : corr,
    'mae_level_test'    : float(mae_level),
    'rmse_level_test'   : rmse_level,
    'r2_level_test'     : float(r2_level),
    'alpha_E_avg'       : float(history['alpha_E'][-1]),
    'beta_volga_avg'    : float(history['beta_volga'][-1]),
    'gamma_other_avg'   : float(history['gamma_other'][-1]),
    'Q_in_m_avg'        : float(history['Q_in_m'][-1]),
    'stopped_epoch'     : len(history['data_loss']),
}
with open(os.path.join(OUT_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


## 13.  Comparison: PINN-1 vs PINN-2 vs PINN-3

PINN-2 and PINN-3 numbers come from the diploma + your previous runs.


In [ ]:
df_compare = pd.DataFrame([
    dict(Model='PINN-1 (Mass balance)',
         MAE  = round(metrics['mae_dh_test'],  4),
         RMSE = round(metrics['rmse_dh_test'], 4),
         R2_dh    = round(metrics['r2_dh_test'],    3),
         R2_level = round(metrics['r2_level_test'], 3)),
    dict(Model='PINN-2 (Budyko)',     MAE=0.0513, RMSE=0.0455, R2_dh=0.252, R2_level='—'),
    dict(Model='PINN-3 (Penman)',     MAE=None,   RMSE=None,   R2_dh=None,  R2_level=None),  # fill from your PINN-3 metrics.json
])
print(df_compare.to_string(index=False))
df_compare.to_csv(os.path.join(OUT_DIR, 'compare_pinns.csv'), index=False)
